In [ ]:
# @title 🚀 FreeFakeStudio — One Click Setup
# @markdown ---
# @markdown ### Configuration
WORKSPACE = "/content/drive/MyDrive/FreeFakeStudio"  # @param {type:"string"}
# @markdown > Persistent storage on Google Drive (models, cache, results)
UPDATE = False  # @param {type:"boolean"}
# @markdown > Pull latest source code from the repository
REPAIR = False  # @param {type:"boolean"}
# @markdown > Force re-download / verify all model files
REPO = "https://github.com/itskrishnamalhotra-stack/FreeFakeStudio.git"  # @param {type:"string"}
# @markdown > Source repository URL (use your fork if you've modified it)
# @markdown ---

import os, subprocess, shutil
from pathlib import Path

# Mount Google Drive
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

# Prepare Workspace
WS = Path(WORKSPACE)
WS.mkdir(parents=True, exist_ok=True)
APP = WS / 'app'

# Clone or update the application
def _clone_fresh():
    if APP.exists():
        shutil.rmtree(str(APP), ignore_errors=True)
    subprocess.run(['git', 'clone', REPO, str(APP)], check=True)
    print(f"Cloned FreeFakeStudio to {APP}")

if not (APP / 'launch.py').exists():
    _clone_fresh()
elif UPDATE:
    # Force reset + pull to avoid merge conflicts
    subprocess.run(['git', '-C', str(APP), 'fetch', '--all'], capture_output=True)
    subprocess.run(['git', '-C', str(APP), 'reset', '--hard', 'origin/main'], capture_output=True)
    print("Updated FreeFakeStudio")

# Safety check — if launch.py still missing, re-clone
if not (APP / 'launch.py').exists():
    print("launch.py missing after update — re-cloning...")
    _clone_fresh()

# Launch (setup + app)
os.environ['FFS_WORKSPACE'] = str(WS)
os.environ['FFS_REPAIR'] = '1' if REPAIR else ''

exec(open(str(APP / 'launch.py')).read())
